# Exercice 1.3: Analyses multivariées et corrélations sur Ames Housing

**Séance 3: Analyses multivariées et corrélations**

---



## Contexte

Jusqu'ici vous avez analysé les variables une par une, puis par paires. C'est nécessaire, mais insuffisant. Les données immobilières d'Ames contiennent 80 variables qui interagissent : la surface habitable influence le prix, mais elle est elle-même corrélée à la qualité, au nombre de pièces, à l'année de construction. Analyser ces relations en isolement peut conduire à des conclusions erronées.

Dans cet exercice, vous allez passer à l'**analyse multivariée** : explorer simultanément les interdépendances entre plusieurs variables, quantifier ces relations avec des métriques adaptées, et développer votre sens critique face aux corrélations: car corrélation n'implique pas causalité.

---



## Objectifs d'apprentissage

- Calculer et comparer les corrélations de Pearson et Spearman, et comprendre quand l'une est préférable à l'autre
- Créer une heatmap avec clustering hiérarchique et interpréter les groupes de variables
- Produire un pair plot conditionnel pour visualiser les relations à travers une variable catégorielle
- Détecter la multicolinéarité avec le VIF et interpréter les seuils
- Calculer une corrélation partielle pour contrôler l'effet d'une variable confondante
- Visualiser des données multivariées avec les coordonnées parallèles
- Identifier des corrélations trompeuses et formuler des explications causales alternatives

---



## Variables utilisées dans cet exercice

| Variable | Description | Type |
|----------|-------------|------|
| `SalePrice` | Prix de vente ($) | Quantitative continue |
| `GrLivArea` | Surface habitable (ft²) | Quantitative continue |
| `LotArea` | Surface du terrain (ft²) | Quantitative continue |
| `TotalBsmtSF` | Surface du sous-sol (ft²) | Quantitative continue |
| `1stFlrSF` | Surface du 1er étage (ft²) | Quantitative continue |
| `YearBuilt` | Année de construction | Quantitative discrète |
| `OverallQual` | Note de qualité (1 à 10) | Ordinale |
| `BldgType` | Type de bâtiment | Qualitative nominale |

Vous serez amenés à explorer d'autres variables numériques du dataset selon vos besoins.

---



## Section 1: Imports et préparation



**Ce que vous devez faire :**

1. Importez les bibliothèques nécessaires. En plus de pandas, numpy, matplotlib et seaborn, cette séance nécessite `scipy`, `pingouin`, et `plotly.express`. Vérifiez qu'elles sont installées.

2. Chargez le dataset et récupérez vos traitements de l'exercice précédent (ou rechargez proprement depuis le fichier CSV), ainsi que le style et la palette choisie durant l'exercice 2.

3. Identifiez toutes les colonnes numériques du dataset.

> `pandas` permet de filter les colonnes d'un DataFrame par type avec la fonction `.select_dtypes()`

4. Pour les étapes suivantes, vous travaillerez avec un **sous-ensemble de 10 à 15 variables numériques pertinentes**. Sélectionnez-les maintenant et justifiez votre choix dans une cellule Markdown. Incluez impérativement `SalePrice`, `GrLivArea`, `LotArea`, `TotalBsmtSF`, `1stFlrSF`, `YearBuilt` et `OverallQual`. Basez-vous sur les analyses faites dans les 2 notebooks précédents, le pourcentage de données manquantes, etc.

5. Supprimez les lignes contenant des valeurs manquantes sur ces variables et conservez ce DataFrame propre pour toute la suite. Documentez le nombre de lignes perdues.



---
## Section 2: Matrice de corrélation et heatmap avec clustering hiérarchique

**Objectif** : obtenir une vue d'ensemble des corrélations entre variables et identifier des groupes de variables liées.



### 2.1 Calcul des deux matrices



**Ce que vous devez faire :**

6. Calculez la matrice de corrélation de **Pearson** sur votre sous-ensemble de variables.

7. Calculez la matrice de corrélation de **Spearman** sur les mêmes variables.

> `df.corr()` calcule Pearson par défaut. Pour Spearman, un paramètre `method` permet de choisir la méthode. Consultez la documentation de `pandas.DataFrame.corr`.

8. Pour chaque matrice, extrayez et n'affichez que les **5 paires de variables** ayant les corrélations les plus fortes (positives) et les **5 paires** les plus négatives, en excluant la diagonale.

> `.unstack()` sur une matrice transforme celle-ci en Série avec un index à deux niveaux. Vous pouvez ensuite trier et filtrer pour extraire les paires extrêmes. Pensez à exclure les paires symétriques (A-B et B-A) ainsi que les diagonales (A-A) pour éviter les doublons.



**Dans une cellule Markdown, répondez :**
- Quelles variables sont les plus avec `SalePrice` selon Pearson ? Selon Spearman ?
- Y a-t-il des paires pour lesquelles les deux coefficients divergent notablement ? Qu'est-ce que cela indique sur la forme de la relation ?



### 2.2 Heatmap avec clustering hiérarchique

**Ce que vous devez faire :**
- Créez une heatmap de la matrice de corrélation de Pearson avec un clustering hiérarchique sur les lignes et les colonnes.

> `sns.clustermap()` fait exactement cela : il réorganise les variables selon leur similarité et ajoute des dendrogrammes. Les paramètres clés sont `figsize`, `cmap` (utilisez une palette **divergente** centrée sur 0: pourquoi ?), `center`, et `linewidths`. 

> Consultez la documentation pour voir comment afficher ou masquer les annotations.



**Dans une cellule Markdown, répondez :**
- Combien de groupes (clusters) de variables distinguez-vous dans le dendrogramme ?
- Décrivez chaque groupe : quelles variables se retrouvent ensemble, et quelle logique métier pourrait expliquer ce regroupement ?
- Quelles variables forment un cluster autour de `SalePrice` ? Est-ce cohérent avec vos attentes ?



- qualité et rénovations: les maisons plus récentes ou rénovées récemment on plus de chance d'être de meilleure qualité


---
## Section 3: Pair plot conditionnel

**Objectif** : visualiser les relations bivariées entre plusieurs variables simultanément, en distinguant les niveaux de qualité.

**Ce que vous devez faire :**
- Sélectionnez **6 variables** pour le pair plot : `SalePrice`, `GrLivArea`, `LotArea`, `YearBuilt`, `TotalBsmtSF`, et `OverallQual`.
- Créez un pair plot coloré par `OverallQual`.

> `sns.pairplot()` avec le paramètre `hue` colore chaque point selon une variable catégorielle. Pour `OverallQual` qui est ordinale (1 à 10), une palette séquentielle comme `"viridis"` est plus appropriée qu'une palette qualitative: pourquoi ? Si le graphique est trop lent à générer ou illisible, utilisez `df.sample(800)` pour réduire le nombre de points.

> Par défaut, la diagonale affiche des KDE par groupe. Pour des histogrammes à la place, regardez le paramètre `diag_kind`.



**Dans une cellule Markdown, répondez :**
- Pour quelles paires de variables la relation semble-t-elle changer selon le niveau de qualité ?
- Les maisons de qualité élevée (≥ 8) forment-elles un groupe visiblement distinct sur le scatterplot `GrLivArea` vs `SalePrice` ?
- Le pair plot vous a-t-il révélé une relation que vous n'aviez pas identifiée dans les exercices précédents ?


---
## Section 4: Détection de la multicolinéarité avec le VIF

**Objectif** : identifier les variables numériques trop redondantes pour être utilisées simultanément dans un modèle de régression.

**Ce que vous devez faire :**

- Calculez le **Variance Inflation Factor (VIF)** pour chacune de vos 10 à 15 variables numériques sélectionnées en Section 1.

> `statsmodels` contient la fonction `variance_inflation_factor` dans le module `statsmodels.stats.outliers_influence`. Cette fonction prend en entrée une **matrice** et l'**indice** de la variable à évaluer. Vous devez donc l'appeler en boucle sur chaque variable. Créez un DataFrame de résultats avec les colonnes `Variable` et `VIF`, trié par VIF décroissant.

> Attention : la fonction nécessite une matrice sans valeurs manquantes. Assurez-vous d'utiliser le DataFrame propre créé en Section 1.

- Visualisez les VIF sous forme de **diagramme en barres horizontal**, avec des lignes verticales de référence à 5 et à 10 pour matérialiser les seuils d'alerte.



**Dans une cellule Markdown, répondez :**
- Quelles variables dépassent le seuil de 5 ? De 10 ?
- Identifiez au moins une paire de variables fortement colinéaires et expliquez la logique métier derrière cette corrélation (pourquoi ces deux variables mesurent-elles en partie la même chose ?).
- Si vous deviez construire un modèle de régression prédisant `SalePrice`, quelles variables supprimeriez-vous ou combiner ? Justifiez.



---
## Section 5: Corrélation partielle

**Objectif** : isoler la relation directe entre `GrLivArea` et `SalePrice` en contrôlant l'effet de `TotalBsmtSF`.



### 5.1 Calcul et comparaison

**Ce que vous devez faire :**
- Calculez la corrélation de **Pearson simple** entre `GrLivArea` et `SalePrice`.
- Calculez la **corrélation partielle** entre `GrLivArea` et `SalePrice`, en contrôlant `TotalBsmtSF`.

> La librairie `pingouin` dispose d'une fonction `partial_corr()` qui prend un DataFrame et les noms des variables `x`, `y`, et `covar` (la variable à contrôler). Elle retourne un DataFrame avec le coefficient `r`, la p-value, et l'intervalle de confiance.

**Dans une cellule Markdown, répondez :**
- Le coefficient de corrélation partielle est-il plus élevé ou plus bas que la corrélation simple ? Qu'est-ce que cela indique sur le rôle de `TotalBsmtSF` ?
- La relation entre surface habitable et prix reste-t-elle significative après contrôle ? (regardez la p-value)
- En une phrase : que vous apprend la corrélation partielle que la corrélation simple ne pouvait pas vous dire ?



### 5.2 Extension

**Ce que vous devez faire :**
- Calculez également la corrélation partielle entre `GrLivArea` et `SalePrice` en contrôlant cette fois `OverallQual`. Comparez les trois coefficients (simple, contrôlé par `TotalBsmtSF`, contrôlé par `OverallQual`) dans un tableau récapitulatif.




**Dans une cellule Markdown, répondez :**
- Quelle variable de contrôle réduit le plus la corrélation entre surface et prix ? Qu'est-ce que cela dit sur les mécanismes sous-jacents ?



## Section 7: Corrélations trompeuses

**Objectif** : développer l'esprit critique face aux corrélations observées.

**Ce que vous devez faire :**
- En vous basant sur vos matrices de corrélation (Pearson ou Spearman), identifiez **trois paires de variables** présentant une corrélation relativement forte (|r| > 0.4) mais pour lesquelles un lien de cause à effet direct semble discutable.

Pour chaque paire, rédigez dans une cellule Markdown :

| Paire de variables | Coefficient observé | Explication alternative | Variable confondante probable |
|-------------------|---------------------|-------------------------|-------------------------------|
| ... | ... | ... | ... |

> Pour trouver des paires intéressantes, pensez aux variables qui mesurent indirectement la même chose (taille, qualité, ancienneté). Une corrélation forte entre deux variables de taille n'est pas nécessairement causale: c'est peut-être une troisième variable (le type de bâtiment, le quartier, la période de construction) qui explique les deux.

**Exemple de raisonnement attendu**: *"Les ventes de glaces et les noyades sont corrélées positivement. Mais acheter une glace ne cause pas les noyades. La variable confondante est la chaleur estivale, qui augmente à la fois la consommation de glaces et la pratique de la baignade."*

- Pour la paire qui vous semble la plus intéressante, créez un scatterplot et annotez-le pour illustrer votre argument.

---



## Checklist avant de rendre

- [ ] Le notebook s'exécute du début à la fin sans erreur (`Restart & Run All`)
- [ ] Le sous-ensemble de variables est justifié en Markdown (Section 1)
- [ ] La clustermap est lisible, avec palette divergente centrée sur 0, et les clusters sont commentés
- [ ] Le pair plot est coloré par `OverallQual` avec une palette adaptée
- [ ] Le tableau VIF est trié par valeur décroissante avec une visualisation barh
- [ ] Les trois coefficients de corrélation partielle sont comparés dans un tableau
- [ ] Les trois corrélations trompeuses sont présentées dans le tableau structuré avec raisonnement
---

